# Social Services — A Cross-Tab That Disproves Its Own Inference

ACS table `B22003` crosses SNAP receipt against poverty status. It is a clean two-by-two, every
margin adds up exactly, and it invites one particular reading: **59.4% of households below the
poverty line do not receive SNAP**, therefore a 59% non-take-up rate.

The same table's other off-diagonal cell says that reading is wrong. **57.0% of households
receiving SNAP are at or above the poverty line** — a majority of recipients. If poverty were
the eligibility criterion, those 8,557,597 households could not be enrolled.

They are enrolled because SNAP's gross-income limit is **130% of the federal poverty
guidelines**, not 100% of the poverty threshold, with asset tests and broad-based categorical
eligibility on top. The poverty line in this table belongs to a different programme. Nothing in
the API says so, and the table proves it anyway.

Both quantities vary widely across states — recipients above poverty from **43.0%**
(Mississippi) to **66.2%** (Maryland); below-poverty non-enrolment from **48.2%** (Rhode
Island) to **75.9%** (Utah) — and they are **uncorrelated** (ρ = **+0.071**, p = **0.618**), so
one cannot be used to adjust the other.

## Learning Objectives

1. **Check whether a cross-tab's second dimension is the criterion you think it is**, before
   reading a rate off it.
2. **Use a table's own off-diagonal cells as a test of your interpretation.**
3. **Distinguish a programme's eligibility rule from a statistical threshold** that shares its
   name and its base.
4. **Report a margin on the share you would actually quote**, and say which independence
   assumption it rests on.

## Prerequisites

- Handbook §3 (denominators, universes and partitions) — the corpus's most frequently confirmed
  failure mode.
- Handbook §1.1 (ACS margins of error), and the proportion formula.
- D01 (income and poverty) for the poverty measure itself.

## Data Sources

**U.S. Census Bureau, ACS 5-year 2023, table `B22003`** — *Receipt of Food Stamps/SNAP in the
Past 12 Months by Poverty Status in the Past 12 Months for Households*, at state level.

Puerto Rico is excluded here, because it does not operate SNAP: it receives the Nutrition
Assistance Program block grant under different rules, so a SNAP-receipt cross-tab is not
comparable.

**What the poverty column is.** The Census poverty threshold, used for statistical measurement.
**What SNAP eligibility is.** Gross monthly income at or below 130% of the federal poverty
*guidelines* — a different series, issued by HHS, for administrative use — plus a net-income
test, an asset test, and categorical eligibility routes that waive parts of both. Those are two
different numbers with similar names, and this table contains only the first.

## 1. Setup

In [1]:
"""Configuration."""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy import stats

import sys
from pathlib import Path

#: Find the package by walking UP from wherever this runs, so the notebook works
#: whether it is executed in place or from the repository root. Sixteen notebooks
#: imported `krl_tutorials_data` with no path discovery and therefore ran only
#: from the root -- which nobody noticed while every run happened to start there.
_repo = next((c for c in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
              if (c / "src" / "krl_tutorials_data").is_dir()), None)
if _repo is None:
    raise SystemExit("krl_tutorials_data not found above " + str(Path.cwd()))
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo / "src"))

from krl_tutorials_data.sources import fetch_acs_wide
from krl_tutorials_data.uncertainty import Estimate, Z90, proportion_moe

YEAR = 2023

#: B22003 -- SNAP receipt crossed with poverty status. Labels verbatim, because
#: the whole notebook is about what the second dimension is and is not.
CROSSTAB = {
    "B22003_001E": "Total households",
    "B22003_002E": "Household received SNAP in the past 12 months",
    "B22003_002M": "Received SNAP, MOE",
    "B22003_003E": "Received SNAP :: Income below poverty level",
    "B22003_003M": "Received SNAP, below poverty, MOE",
    "B22003_004E": "Received SNAP :: Income at or above poverty level",
    "B22003_004M": "Received SNAP, at or above poverty, MOE",
    "B22003_005E": "Household did not receive SNAP",
    "B22003_006E": "Did not receive SNAP :: Income below poverty level",
    "B22003_006M": "Did not receive SNAP, below poverty, MOE",
    "B22003_007E": "Did not receive SNAP :: Income at or above poverty level",
}

#: SNAP gross-income eligibility is 130% of the federal poverty guidelines in most
#: states, with asset tests and broad-based categorical eligibility on top. The
#: poverty line in this table is a DIFFERENT threshold from a different programme.
#: Nothing in the API says so, and section 4 shows the table disproving the
#: conflation by itself.
SNAP_GROSS_INCOME_LIMIT = "130% of the federal poverty guidelines (most states)"

pd.set_option("display.width", 112)
print(f"vintage : ACS 5-year {YEAR}, state level")
print(f"table   : B22003, SNAP receipt x poverty status")
print(f"note    : SNAP gross-income limit is {SNAP_GROSS_INCOME_LIMIT}")


vintage : ACS 5-year 2023, state level
table   : B22003, SNAP receipt x poverty status
note    : SNAP gross-income limit is 130% of the federal poverty guidelines (most states)


## 2. Retrieval, and the margins

Check that the cross-tabulation adds up before reading anything off it. A table whose margins do
not reconcile cannot support a rate.

In [2]:
"""Retrieve and check the cross-tabulation adds up."""
households, prov = fetch_acs_wide(year=YEAR, variables=CROSSTAB, geography="state:*")
households = households[households["NAME"] != "Puerto Rico"].reset_index(drop=True)

cell = {n: households[f"B22003_{n:03d}E"] for n in range(1, 8)}
checks = {
    "received + did not == total": cell[2] + cell[5] - cell[1],
    "SNAP rows == SNAP total": cell[3] + cell[4] - cell[2],
    "non-SNAP rows == non-SNAP total": cell[6] + cell[7] - cell[5],
}
print(f"states: {len(households)}")
for label, residual in checks.items():
    print(f"  {label:<34} exact in {int((residual == 0).sum())} of {len(households)}")
print(f"payload sha256: {prov.content_sha256[:16]}...")


states: 51
  received + did not == total        exact in 51 of 51
  SNAP rows == SNAP total            exact in 51 of 51
  non-SNAP rows == non-SNAP total    exact in 51 of 51
payload sha256: e6db631f3bf503a5...


**All three identities hold exactly, in 51 of 51 states.** The table is internally
consistent, which is necessary and — as D26 showed at length — nowhere near sufficient.

## 3. The two-by-two

In [3]:
"""The two-by-two, nationally."""
grid = pd.DataFrame(
    [[cell[3].sum(), cell[4].sum()], [cell[6].sum(), cell[7].sum()]],
    index=["received SNAP", "did not receive SNAP"],
    columns=["below poverty", "at or above poverty"],
).astype("int64")
grid["total"] = grid.sum(axis=1)
grid.loc["total"] = grid.sum()
print(grid.to_string(formatters={c: "{:,.0f}".format for c in grid.columns}))
print()
print(f"households in the table : {int(cell[1].sum()):,}")


                     below poverty at or above poverty       total
received SNAP            6,447,353           8,557,597  15,004,950
did not receive SNAP     9,432,695         103,045,220 112,477,915
total                   15,880,048         111,602,817 127,482,865

households in the table : 127,482,865


Read the top-right cell before the bottom-left one. **8,557,597 households receive
SNAP while being at or above the poverty line**, against 6,447,353 who receive it while below.

That is the whole argument, and it is visible before any rate is computed.

## 4. What each off-diagonal cell means

In [4]:
"""The cell that settles it."""
snap_total = cell[2].sum()
above_share = cell[4].sum() / snap_total
below_not_enrolled = cell[6].sum() / (cell[3].sum() + cell[6].sum())

print(f"SNAP households at or above the poverty line : {int(cell[4].sum()):,} "
      f"of {int(snap_total):,}")
print(f"  as a share of all SNAP households          : {above_share:.1%}")
print()
print(f"below-poverty households not receiving SNAP  : {int(cell[6].sum()):,} "
      f"of {int(cell[3].sum() + cell[6].sum()):,}")
print(f"  as a share of below-poverty households     : {below_not_enrolled:.1%}")


SNAP households at or above the poverty line : 8,557,597 of 15,004,950
  as a share of all SNAP households          : 57.0%

below-poverty households not receiving SNAP  : 9,432,695 of 15,880,048
  as a share of below-poverty households     : 59.4%


**57.0% of SNAP households are at or above the poverty line.**

A majority of the programme's recipients sit above the threshold this table crosses them
against. That is not an anomaly, an error, or fraud: SNAP's gross-income limit is 130% of the
federal poverty guidelines, there is a separate net-income test after deductions, and
broad-based categorical eligibility admits households through other means-tested programmes
entirely. A household at 120% of poverty is squarely eligible.

So the other cell — **59.4% of below-poverty households not receiving SNAP** — cannot be read as
a non-take-up rate. Its denominator is not the eligible population. It is too narrow in one
direction (it excludes eligible households between 100% and 130% of poverty) and too broad in
another (it includes below-poverty households failing the asset test or excluded by immigration
status or student rules).

**[rule]** Before computing a rate from a cross-tab, ask whether the conditioning variable is
the programme's criterion or a statistic that resembles it. If the table's own off-diagonal cell
is large, it is not the criterion, and the table has just told you so.

## 5. Both vary, and they do not move together

In [5]:
"""Both quantities vary a great deal, and independently."""
above = 100 * cell[4] / cell[2]
unenrolled = 100 * cell[6] / (cell[3] + cell[6])
# .to_numpy() on purpose: `above` and `unenrolled` carry the frame's integer
# index, and passing `households["NAME"]` as the index of a DataFrame built from
# them realigns on that integer index and yields an all-NA table. The first draft
# of this cell did exactly that and printed `nan` for every state.
summary = pd.DataFrame({
    "SNAP recipients above poverty %": above.to_numpy(float),
    "below-poverty not on SNAP %": unenrolled.to_numpy(float),
}, index=households["NAME"].to_numpy())

for label in summary.columns:
    series = summary[label]
    print(f"{label}")
    print(f"  median {series.median():.1f}%   "
          f"low {series.min():.1f}% ({series.idxmin()})   "
          f"high {series.max():.1f}% ({series.idxmax()})")
rho, pval = stats.spearmanr(above, unenrolled)
print()
print(f"Spearman between the two = {rho:+.3f}  p = {pval:.3f}  n = {len(above)}")


SNAP recipients above poverty %
  median 55.8%   low 43.0% (Mississippi)   high 66.2% (Maryland)
below-poverty not on SNAP %
  median 61.3%   low 48.2% (Rhode Island)   high 75.9% (Utah)

Spearman between the two = +0.071  p = 0.618  n = 51


The share of recipients above poverty runs from **43.0%** in Mississippi to
**66.2%** in Maryland. The share of below-poverty households not enrolled runs from **48.2%** in
Rhode Island to **75.9%** in Utah. Both are wide enough that a national figure describes no
particular state.

And the two are **uncorrelated across states** — ρ = **+0.071**, p = **0.618**. That matters
practically. If they moved together, a reader might use the above-poverty share to deflate the
non-enrolment share and recover something closer to take-up. They do not, so that correction is
not available, and the honest position is that this table does not measure take-up at all.

**[rule]** When a measure is known to be biased and the bias is measurable elsewhere in the same
table, check whether the two are related before proposing an adjustment. An uncorrelated bias
cannot be corrected for — it can only be reported.

## 6. The margin on the number a reader would quote

In [6]:
"""Margins, on the share a reader would actually quote."""
moe = np.array([
    proportion_moe(
        Estimate(value=households.loc[i, "B22003_006E"], moe=households.loc[i, "B22003_006M"]),
        Estimate(value=households.loc[i, "B22003_003E"] + households.loc[i, "B22003_006E"],
                 moe=float(np.hypot(households.loc[i, "B22003_003M"],
                                    households.loc[i, "B22003_006M"]))),
    )
    for i in households.index
])
value = (cell[6] / (cell[3] + cell[6])).to_numpy(float)
print(f"below-poverty-not-on-SNAP share, median {100 * np.median(value):.1f}%")
print(f"  median 90% MOE            : +/-{100 * np.median(moe):.2f} pp")
print(f"  widest                    : +/-{100 * moe.max():.2f} pp "
      f"({households.loc[int(np.argmax(moe)), 'NAME']})")
print()
print("The denominator MOE is combined in quadrature, which assumes the two cells'")
print("errors are independent. The Bureau does not publish that covariance, so the")
print("figure above is the independent-assumption case and is stated as such")
print("(handbook 1.1).")


below-poverty-not-on-SNAP share, median 61.3%
  median 90% MOE            : +/-0.84 pp
  widest                    : +/-2.42 pp (Delaware)

The denominator MOE is combined in quadrature, which assumes the two cells'
errors are independent. The Bureau does not publish that covariance, so the
figure above is the independent-assumption case and is stated as such
(handbook 1.1).


The share carries a **median 90% margin of ±0.84 percentage points** at state level,
widening to **±2.42** in Delaware. That precision is irrelevant to
the problem: the quantity is precisely estimated and is not the quantity anyone wants.

Note also what the denominator's margin assumes. Combining two cells' margins in quadrature
assumes their errors are independent, and the Bureau publishes no covariance for ACS cells. The
figure is therefore the independent-assumption case, stated as such per handbook §1.1 — the same
caveat D07 reports for age standardisation.

## Limitations

1. **This notebook does not measure SNAP take-up.** It establishes that this table cannot, and
   proposes no substitute. A real take-up estimate needs an eligibility model over microdata,
   which ACS summary tables cannot supply.

2. **"Received SNAP in the past 12 months" is not current enrolment.** A household enrolled for
   one month in the period is counted identically to one enrolled throughout.

3. **Poverty status in ACS is measured over the same twelve months**, so both dimensions are
   annual and neither is a point-in-time state. A household can be below poverty in the months it
   was not enrolled and above it in the months it was.

4. **The 130% figure is the gross-income test in most states.** States operating broad-based
   categorical eligibility use higher limits — up to 200% in some — so the mismatch this notebook
   documents is larger in those states and the notebook does not measure by how much.

5. **Puerto Rico is excluded** because it operates NAP rather than SNAP. Other territories are
   not in the `state:*` universe at all.

6. **State-level only.** Nothing here is a county or household claim, and the ecological
   inference is not made.

7. **The margin in section 6 rests on an independence assumption** the Bureau does not publish a
   covariance for.

## Key Takeaways

1. **A table can be internally perfect and still not support the rate you want.** All three
   identities in `B22003` hold in 51 of 51 states.

2. **57.0% of SNAP households are at or above the poverty line** — 8,557,597 of them.

3. **Therefore the 59.4% figure is not a non-take-up rate.** Its denominator is neither the
   eligible population nor a subset of it.

4. **SNAP eligibility is 130% of the federal poverty guidelines**, plus net-income, asset and
   categorical routes. The poverty threshold in this table is a different number for a different
   purpose.

5. **Use the off-diagonal cell as the test.** If it is large, the conditioning variable is not
   the criterion.

6. **The two shares are uncorrelated** (ρ = +0.071, p = 0.618), so no adjustment is available —
   only disclosure.

7. **Precision is not relevance.** The share is tightly estimated and is still the wrong
   quantity.

## Exercises

1. **Bound the eligible population.** Using `B17002` (ratio of income to poverty level), compute
   households under 130% of poverty by state and recompute section 4's share against that
   denominator. State precisely what the new figure still does not account for.

2. **Find the categorical-eligibility states.** Identify which states operate broad-based
   categorical eligibility above 130%, and test whether their above-poverty recipient share is
   higher. Report the test you ran and its family size.

3. **Test the independence assumption.** Recompute section 6's margins under perfect positive
   correlation between the two cells and report how much the interval widens, following D07's
   construction.

4. **Look for the same structure elsewhere.** Find another ACS table that crosses programme
   receipt against a statistical threshold, and check whether its off-diagonal cell carries the
   same warning.

5. **Write the correction.** A published report states "59% of poor households do not receive
   SNAP." Draft the two-sentence correction you would send, using only figures from this table.

6. **Decide what you would publish.** Given everything here, state the one number about SNAP and
   poverty you would put in a brief, with its caveat, in under forty words.

## References

- U.S. Census Bureau. *ACS table B22003: Receipt of Food Stamps/SNAP in the Past 12 Months by
  Poverty Status.* https://data.census.gov/table/ACSDT5Y2023.B22003
- USDA Food and Nutrition Service. *SNAP Eligibility.*
  https://www.fns.usda.gov/snap/recipient/eligibility
- U.S. Department of Health and Human Services. *Poverty Guidelines* — the administrative series
  SNAP's 130% test is applied to, distinct from the Census poverty thresholds.
  https://aspe.hhs.gov/topics/poverty-economic-mobility/poverty-guidelines
- U.S. Census Bureau. *How the Census Bureau Measures Poverty.*
  https://www.census.gov/topics/income-poverty/poverty/guidance/poverty-measures.html
- Handbook §1.1 (margins of error), §3 (denominators, universes and partitions).